# Notebook 2 — Chunking and Embedding

This notebook takes the raw 3GPP PDF pages from Notebook 1 and prepares them
for storage in ChromaDB. We split each page into overlapping chunks so the
retrieval system can return precise, self-contained passages rather than entire
pages. We then convert each chunk into a vector embedding using OpenAI and
store everything locally in ChromaDB.

In [1]:
import sys
from dotenv import load_dotenv
import os

load_dotenv()

print(sys.executable)  # confirm rag3gpp environment
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))

c:\Users\nabee\anaconda3\envs\rag3gpp\python.exe
OpenAI key loaded: True


## Loading and chunking the documents

We load all three PDFs and split them into overlapping chunks using LangChain's
RecursiveCharacterTextSplitter. The chunk size of 1000 characters balances
context richness against retrieval precision — large enough to capture a full
technical clause, small enough that the LLM isn't flooded with irrelevant text.
The 200 character overlap prevents information loss at chunk boundaries.

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# load all three specs
pdf_files = [
    r"C:\Projects\3gpp-rag\data\raw\TS_38211.pdf",
    r"C:\Projects\3gpp-rag\data\raw\TS_38214.pdf",
    r"C:\Projects\3gpp-rag\data\raw\TS_38300.pdf",
]

all_pages = []
for path in pdf_files:
    loader = PyPDFLoader(path)
    all_pages.extend(loader.load())

print(f"Total pages loaded: {len(all_pages)}")

# split pages into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]  # tries paragraph breaks first, then line breaks
)

chunks = splitter.split_documents(all_pages)

print(f"Total chunks created: {len(chunks)}")
print(f"\nExample chunk:\n{chunks[10].page_content}")
print(f"\nExample chunk metadata: {chunks[10].metadata}")

Total pages loaded: 841
Total chunks created: 3697

Example chunk:
5.2.1 Pseudo-random sequence generation ......................................................................................................... 18 
5.2.2 Low-PAPR sequence generation type 1 ..................................................................................................... 18 
5.2.2.1 Base sequences of length 36 or larger .................................................................................................. 18 
5.2.2.2 Base sequences of length less than 36 .................................................................................................. 19 
5.2.3 Low-PAPR sequence generation type 2 ..................................................................................................... 22 
5.2.3.1 Sequences of length 30 or larger .......................................................................................................... 22

Example chunk metadata: {'producer': 

## Generating embeddings and storing in ChromaDB

We convert each chunk into a 1536-dimensional vector using OpenAI's
text-embedding-3-small model, then store the vectors locally in ChromaDB.
This is a one-time operation — once the database is built we query it
without re-embedding. We use a persistent ChromaDB client so the vectors
survive between sessions.

In [4]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# set up the embedding model
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# this will embed all 3697 chunks and store them to disk
# takes 3-5 minutes and costs roughly $0.05
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=r"C:\Projects\3gpp-rag\chroma_db"
)

print(f"Chunks stored in ChromaDB: {vectorstore._collection.count()}")

Chunks stored in ChromaDB: 3697


## Testing retrieval

A quick sanity check to confirm ChromaDB can take a natural language query,
find the most semantically similar chunks, and return them with their source
metadata. This is the core operation the RAG chain will rely on.

In [5]:
# load the existing vectorstore from disk (no re-embedding needed)
vectorstore = Chroma(
    persist_directory=r"C:\Projects\3gpp-rag\chroma_db",
    embedding_function=embeddings
)

# test query relevant to the specs we loaded
query = "How is the physical downlink shared channel modulated in NR?"

results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata.get('title')} | Page: {doc.metadata.get('page')}")
    print(f"Content: {doc.page_content[:300]}")
    print()

--- Result 1 ---
Source: 3GPP TS 38.214 | Page: 36
Content: used in the physical downlink shared channel. 
elseif the UE is configured with the higher layer parameter mcs-Table given by SPS-Config set to 'qam64LowSE' 
- if the PDSCH is scheduled by a PDCCH with CRC scrambled by CS -RNTI or 
- if the PDSCH is scheduled without corresponding PDCCH transmission

--- Result 2 ---
Source: 3GPP TS 38.300 | Page: 56
Content: RAN; 
- support for HARQ; 
- support for dynamic link adaptation by varying the transmit power, modulation and coding ; 
- support for SL discontinuous reception (SL DRX) to enable UE power saving.  
Association of transport channels to physical channels is described in TS 38.202 [ 20]. 
5.6 Access 

--- Result 3 ---
Source: 3GPP TS 38.211 | Page: 4
Content: 7.2 Physical resources .......................................................................................................................................... 118 
7.3 Physical channels ............................